# Lab 06 Web Scrapping

In [39]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import ast
from pandas import json_normalize
from tqdm import tqdm
import folium

## Starbucks store locator

- Website: https://www.starbucks.co.uk/store-locator?types=starbucks&latLng=55.8625388%2C-4.284226000000002&zoom=12

- Investigate the website and the feature of the URLs.
- Get the latitudes, longitudes, addresses, Unique IDs, store names, and openning hours of all the Starbucks in the area below as a DataFrame. 
- Visualise the stores on map. 

The boudanries to scrap:

- north_bound = 60.8590 N
- south_bound = 54.6356 N
- west_bound = -7.385 W
- east_bound = 1.7834 E

In [40]:
starbucks_df = pd.DataFrame()

# Base URL for the API
base_url = 'https://www.starbucks.co.uk/api/v2/stores/'

# Just trying with Glasgow area coordinates
params = {
    'filter[coordinates][latitude]': 55.8642,
    'filter[coordinates][longitude]': -4.2518,
    'filter[radius]': 20  # 20km radius around Glasgow
}

response = requests.get(base_url, params=params)
response.raise_for_status()  
data = response.json()

# Convert to DataFrame
starbucks_df = json_normalize(data['data'])

print(len(starbucks_df))



41


In [41]:
starbucks_df.head(10)

,id,type,attributes.storeNumber,attributes.name,attributes.address.streetAddressLine1,attributes.address.streetAddressLine2,attributes.address.streetAddressLine3,attributes.address.city,attributes.address.countrySubdivisionCode,attributes.address.countryCode,...,attributes.todayHours.open24Hours,attributes.todayHours.openAsOfLocalTime,attributes.todayHours.openTime,attributes.todayHours.closeTime,attributes.todayHours.opensIn,attributes.todayHours.closesIn,attributes.todayHours.localTime,attributes.timeZoneInfo.currentTimeOffset,attributes.timeZoneInfo.windowsTimeZoneId,attributes.timeZoneInfo.olsonTimeZoneId
0,2991,store,12784-139662,Glasgow - Buchanan Galleries -,Glasgow Buchanan Galleries,Buchanan Galleries,"Unit 15, Level 4",Glasgow,SCT,GB,...,False,False,08:00:00,18:00:00,12:41:36,None,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
1,2674,store,12053-10195,Glasgow-Sauchiehall Street,27 Sauchiehall St,None,None,Glasgow,SCT,GB,...,False,True,06:30:00,21:00:00,None,1:41:36,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
2,2751,store,12562-60620,Glasgow - Buchanan Street,136 140 Buchanan Street,None,None,Glasgow,SCT,GB,...,False,True,06:30:00,22:00:00,None,2:41:36,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
3,2823,store,12836-137777,Glasgow - Nelson Mandela Squar,Glasgow West Nile Street,None,58 Nelson Mandela Square,Glasgow,SCT,GB,...,False,False,07:00:00,19:00:00,11:41:36,None,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
4,1005532,store,18489-191425,Glasgow - Central Station,1 Gordon Street,Glasgow Central Station,None,Glasgow,SCT,GB,...,False,True,06:00:00,20:00:00,None,0:41:36,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
5,2554,store,12045-10176,Glasgow-Bothwell Street,33 Bothwell St,Flat Basement,None,Glasgow,SCT,GB,...,False,False,06:00:00,18:00:00,10:41:36,None,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
6,1017467,store,47633-306329,Glasgow-St Enochs,64 Dunlop Street,None,None,Glasgow,SCT,GB,...,False,True,08:00:00,20:00:00,None,0:41:36,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
7,1009199,store,22808-222385,Glasgow - Village Hotel,Village Hotels Starbucks,None,Festival Gate,Glasgow,SCT,GB,...,False,False,07:00:00,15:00:00,11:41:36,None,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
8,1018522,store,50355-256560,Glasgow - Forge RP DT,Biggar Street,None,None,Glasgow,SCT,GB,...,False,True,06:00:00,21:00:00,None,1:41:36,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London
9,2791,store,12166-24311,Glasgow - Byres Road,252 254 Byres Road,None,None,Glasgow,SCT,GB,...,False,True,06:30:00,20:00:00,None,0:41:36,2026-02-16T19:18:23.5014598,0,GMT Standard Time,GMT+01:00 Europe/London


In [42]:
starbucks_df = starbucks_df[['id', 'attributes.name', 'attributes.address.streetAddressLine1', 'attributes.address.streetAddressLine2', 'attributes.address.streetAddressLine3', 'attributes.todayHours.openTime', 'attributes.todayHours.closeTime']]

In [43]:
starbucks_df.head(10)

,id,attributes.name,attributes.address.streetAddressLine1,attributes.address.streetAddressLine2,attributes.address.streetAddressLine3,attributes.todayHours.openTime,attributes.todayHours.closeTime
0,2991,Glasgow - Buchanan Galleries -,Glasgow Buchanan Galleries,Buchanan Galleries,"Unit 15, Level 4",08:00:00,18:00:00
1,2674,Glasgow-Sauchiehall Street,27 Sauchiehall St,None,None,06:30:00,21:00:00
2,2751,Glasgow - Buchanan Street,136 140 Buchanan Street,None,None,06:30:00,22:00:00
3,2823,Glasgow - Nelson Mandela Squar,Glasgow West Nile Street,None,58 Nelson Mandela Square,07:00:00,19:00:00
4,1005532,Glasgow - Central Station,1 Gordon Street,Glasgow Central Station,None,06:00:00,20:00:00
5,2554,Glasgow-Bothwell Street,33 Bothwell St,Flat Basement,None,06:00:00,18:00:00
6,1017467,Glasgow-St Enochs,64 Dunlop Street,None,None,08:00:00,20:00:00
7,1009199,Glasgow - Village Hotel,Village Hotels Starbucks,None,Festival Gate,07:00:00,15:00:00
8,1018522,Glasgow - Forge RP DT,Biggar Street,None,None,06:00:00,21:00:00
9,2791,Glasgow - Byres Road,252 254 Byres Road,None,None,06:30:00,20:00:00


In [44]:
starbucks_df = starbucks_df.rename(columns={
    'id': 'Store ID',
    'attributes.name': 'Store Name',
    'attributes.address.streetAddressLine1': 'Address Line 1',
    'attributes.address.streetAddressLine2': 'Address Line 2',
    'attributes.address.streetAddressLine3': 'Address Line 3',
    'attributes.todayHours.openTime': 'Open Time',
    'attributes.todayHours.closeTime': 'Close Time'
})

starbucks_df.head()

,Store ID,Store Name,Address Line 1,Address Line 2,Address Line 3,Open Time,Close Time
0,2991,Glasgow - Buchanan Galleries -,Glasgow Buchanan Galleries,Buchanan Galleries,"Unit 15, Level 4",08:00:00,18:00:00
1,2674,Glasgow-Sauchiehall Street,27 Sauchiehall St,None,None,06:30:00,21:00:00
2,2751,Glasgow - Buchanan Street,136 140 Buchanan Street,None,None,06:30:00,22:00:00
3,2823,Glasgow - Nelson Mandela Squar,Glasgow West Nile Street,None,58 Nelson Mandela Square,07:00:00,19:00:00
4,1005532,Glasgow - Central Station,1 Gordon Street,Glasgow Central Station,None,06:00:00,20:00:00


In [45]:
starbucks_df['Open Time'] = pd.to_datetime(starbucks_df['Open Time'])
starbucks_df['Close Time'] = pd.to_datetime(starbucks_df['Close Time'])
starbucks_df['Opening Hours'] = starbucks_df['Open Time'].dt.strftime('%H:%M') + ' - ' + starbucks_df['Close Time'].dt.strftime('%H:%M')
starbucks_df.head()

C:\Users\caras\AppData\Local\Temp\ipykernel_14220\4253784822.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  starbucks_df['Open Time'] = pd.to_datetime(starbucks_df['Open Time'])
C:\Users\caras\AppData\Local\Temp\ipykernel_14220\4253784822.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  starbucks_df['Close Time'] = pd.to_datetime(starbucks_df['Close Time'])


,Store ID,Store Name,Address Line 1,Address Line 2,Address Line 3,Open Time,Close Time,Opening Hours
0,2991,Glasgow - Buchanan Galleries -,Glasgow Buchanan Galleries,Buchanan Galleries,"Unit 15, Level 4",2026-02-16 08:00:00,2026-02-16 18:00:00,08:00 - 18:00
1,2674,Glasgow-Sauchiehall Street,27 Sauchiehall St,None,None,2026-02-16 06:30:00,2026-02-16 21:00:00,06:30 - 21:00
2,2751,Glasgow - Buchanan Street,136 140 Buchanan Street,None,None,2026-02-16 06:30:00,2026-02-16 22:00:00,06:30 - 22:00
3,2823,Glasgow - Nelson Mandela Squar,Glasgow West Nile Street,None,58 Nelson Mandela Square,2026-02-16 07:00:00,2026-02-16 19:00:00,07:00 - 19:00
4,1005532,Glasgow - Central Station,1 Gordon Street,Glasgow Central Station,None,2026-02-16 06:00:00,2026-02-16 20:00:00,06:00 - 20:00


In [ ]:
starbucks_df['Address'] = starbucks_df.apply(         #.apply runs function on each row of df
    lambda row: ', '.join(filter(      #join combines strings with specified separator
        lambda x: x and x != 'None',
        [row['Address Line 3'], row['Address Line 2'], row['Address Line 1']] #lambda is filtering rows where address 1,2,3 have a value and aren't None
    )),
    axis=1 #applies to rows
)
            
starbucks_df.head()

,Store ID,Store Name,Address Line 1,Address Line 2,Address Line 3,Open Time,Close Time,Opening Hours,Address
0,2991,Glasgow - Buchanan Galleries -,Glasgow Buchanan Galleries,Buchanan Galleries,"Unit 15, Level 4",2026-02-16 08:00:00,2026-02-16 18:00:00,08:00 - 18:00,"Unit 15, Level 4, Buchanan Galleries, Glasgow ..."
1,2674,Glasgow-Sauchiehall Street,27 Sauchiehall St,None,None,2026-02-16 06:30:00,2026-02-16 21:00:00,06:30 - 21:00,27 Sauchiehall St
2,2751,Glasgow - Buchanan Street,136 140 Buchanan Street,None,None,2026-02-16 06:30:00,2026-02-16 22:00:00,06:30 - 22:00,136 140 Buchanan Street
3,2823,Glasgow - Nelson Mandela Squar,Glasgow West Nile Street,None,58 Nelson Mandela Square,2026-02-16 07:00:00,2026-02-16 19:00:00,07:00 - 19:00,"58 Nelson Mandela Square, Glasgow West Nile St..."
4,1005532,Glasgow - Central Station,1 Gordon Street,Glasgow Central Station,None,2026-02-16 06:00:00,2026-02-16 20:00:00,06:00 - 20:00,"Glasgow Central Station, 1 Gordon Street"


In [ ]:
#no lat and lon for visualisation 
